## Creating frequency lists of elative construction instances in Estonian Reference Corpus

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("..")

import os
import csv

from collections import Counter

from methods.helper_methods import read_texts_from_json

In [ ]:
SENTENCES = "../data/koondkorpus_sentences/conx_example_sentences_20042026/"

In [ ]:
sentences = read_texts_from_json(SENTENCES)

In [88]:
len(sentences)

330847

### Finding construction instances and their frequencies from sentences

#### I collecting construction instances from data

In [ ]:
conx_phrases = []

for idx1, text in enumerate(sentences):
    for stanza_word in text.v172_stanza_syntax:
        if stanza_word.deprel == "nmod":
            if stanza_word.morph_analysis.form[0] == "sg el" and stanza_word.morph_analysis.partofspeech[0] == "S":
                if stanza_word.parent_span and stanza_word.parent_span.partofspeech[0] == "S" and stanza_word.parent_span.id > stanza_word.id:
                    conx_phrases.append([stanza_word, stanza_word.parent_span])

In [90]:
len(conx_phrases)

123877

#### II lemmatizing phrases, finding frequencies

In [ ]:
# variant A: lemmatizing phrase (phrase root is lemmatized)
# "lendurist abikaasa", "siidist kleit", ...
conx_phrase_lemmas = []

for phrase in conx_phrases:
    form_normalized = phrase[1].morph_analysis.form[0] if len(phrase[1].morph_analysis.form[0].split()) < 2 else phrase[1].morph_analysis.form[0].split()[1]  
    conx_phrase_lemmas.append(f"{phrase[0].text.lower()}##{phrase[1].morph_analysis.lemma[0]}##{form_normalized}")

In [ ]:
# variant B: lemmatizing phrase (all phrase members are to be lemmatized)
# "lendur abikaasa", "siid kleit", ...
conx_phrase_lemmas2 = []

for phrase in conx_phrases:
    form_normalized = phrase[1].morph_analysis.form[0] if len(phrase[1].morph_analysis.form[0].split()) < 2 else phrase[1].morph_analysis.form[0].split()[1]  
    conx_phrase_lemmas2.append(f"{phrase[0].morph_analysis.lemma[0]}##{phrase[1].morph_analysis.lemma[0]}##{form_normalized}")

In [101]:
conx_phrase_lemmas[:5]

['pronksist##kuju##kom',
 'veteranist##välisminister##n',
 'seisundist##arusaamine##n',
 'floristist##ettevõtja##n',
 'lasteaiast##saadik##n']

In [104]:
conx_phrase_lemmas2[:5]

['pronks##kuju##kom',
 'veteran##välisminister##n',
 'seisund##arusaamine##n',
 'florist##ettevõtja##n',
 'lasteaed##saadik##n']

In [93]:
conx_phrase_lemma_freqs = Counter(conx_phrase_lemmas).most_common()

In [105]:
conx_phrase_lemma_freqs2 = Counter(conx_phrase_lemmas2).most_common()

In [102]:
conx_phrase_lemma_freqs[:5]

[('plika_tartust##afk##n', 2189),
 ('isikust##ettevõtja##n', 895),
 ('isikust##ettevõtja##g', 673),
 ('isikust##ettevõtja##es', 424),
 ('ametist##vabastamine##g', 361)]

In [106]:
conx_phrase_lemma_freqs2[:5]

[('plikatartu##afk##n', 2189),
 ('isik##ettevõtja##n', 897),
 ('isik##ettevõtja##g', 673),
 ('isik##ettevõtja##es', 424),
 ('amet##vabastamine##g', 361)]

#### III assembling frequency list data

In [95]:
conx_phrase_data = []

for el in conx_phrase_lemma_freqs:
    phrase_and_case = el[0].split("##")

    if len(phrase_and_case) != 3:
        continue

    freq = el[1]
    
    dct = {"member_1": phrase_and_case[0],
           "member_2": phrase_and_case[1],
           "phrase_case": phrase_and_case[2],
           "frequency": freq}
    
    conx_phrase_data.append(dct)
    

In [107]:
conx_phrase_data2 = []

for el in conx_phrase_lemma_freqs2:
    phrase_and_case = el[0].split("##")

    if len(phrase_and_case) != 3:
        continue

    freq = el[1]
    
    dct = {"member_1": phrase_and_case[0],
           "member_2": phrase_and_case[1],
           "phrase_case": phrase_and_case[2],
           "frequency": freq}
    
    conx_phrase_data2.append(dct)

In [96]:
conx_phrase_data[:5]

[{'member_1': 'plika_tartust',
  'member_2': 'afk',
  'phrase_case': 'n',
  'frequency': 2189},
 {'member_1': 'isikust',
  'member_2': 'ettevõtja',
  'phrase_case': 'n',
  'frequency': 895},
 {'member_1': 'isikust',
  'member_2': 'ettevõtja',
  'phrase_case': 'g',
  'frequency': 673},
 {'member_1': 'isikust',
  'member_2': 'ettevõtja',
  'phrase_case': 'es',
  'frequency': 424},
 {'member_1': 'ametist',
  'member_2': 'vabastamine',
  'phrase_case': 'g',
  'frequency': 361}]

In [108]:
conx_phrase_data2[:5]

[{'member_1': 'plikatartu',
  'member_2': 'afk',
  'phrase_case': 'n',
  'frequency': 2189},
 {'member_1': 'isik',
  'member_2': 'ettevõtja',
  'phrase_case': 'n',
  'frequency': 897},
 {'member_1': 'isik',
  'member_2': 'ettevõtja',
  'phrase_case': 'g',
  'frequency': 673},
 {'member_1': 'isik',
  'member_2': 'ettevõtja',
  'phrase_case': 'es',
  'frequency': 424},
 {'member_1': 'amet',
  'member_2': 'vabastamine',
  'phrase_case': 'g',
  'frequency': 361}]

In [ ]:
with open('../data/results/frequency_lists/elative_construction_freqs.csv', 'w', encoding='utf-8', newline='') as csvfile:
    fieldnames = ['member_1', 'member_2', 'phrase_case', 'frequency']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(conx_phrase_data)

In [ ]:
with open('../data/results/frequency_lists/elative_construction_lemma_freqs.csv', 'w', encoding='utf-8', newline='') as csvfile:
    fieldnames = ['member_1', 'member_2', 'phrase_case', 'frequency']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(conx_phrase_data2)